# SRL supplement: figures and tables

Makes every figure and generated table in `pinatubo_srl_supplement.tex` and writes them to `figures/` and `tables/` beside this notebook. Nothing there is edited by hand. Rerun after the conversion and analysis notebooks (or with `python run_notebooks.py --figures`), then build the PDF with `latexmk -pdf pinatubo_srl_supplement.tex`.

| Item | Made here from |
|---|---|
| Figure S1, telemetry | step 70 `diagrams(notes=False)` |
| Table S1, radio-frequency maps | step 68 radio-map records |
| Table S2 and the verbatim control file | `10_conversion/hypo71_settings_199106.txt` |
| Table S3, clock rules | `10_conversion/clock_regimes.csv` |
| Figure S2, EVNT against triggers | step 61 EVNT records and the step 10 waveform index |
| Figures S3–S4, clock anchors | step 72 `plot_records` and its matches |
| Figure S5, waveform availability by channel | analysis run trace inventory |
| Figures S6–S8, BOB availability by series | steps 61–67 record tables (UTC) |
| Figure S9, the two hypocenter summaries | step 43 origin index |
| Figures S10–S11, pick and catalog QC | copies of step 23 and 53 outputs |
| Figures S12–S13, analysis | copies of analysis run figures |

## Configuration

In [1]:
from pathlib import Path
import json, re, shutil, string, sys, tempfile

import matplotlib
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SUPP = next(p for p in [Path.cwd(), Path.cwd() / 'papers/SRL/supplement'] if (p / 'pinatubo_srl_supplement.tex').is_file())
REPO = SUPP.parents[2]
PIPELINE = REPO / '10_conversion'
FIG, TAB = SUPP / 'figures', SUPP / 'tables'
FIG.mkdir(exist_ok=True); TAB.mkdir(exist_ok=True)

sys.path.insert(0, str(PIPELINE))
from lib.config import pipeline_paths
P = pipeline_paths()
UTC = Path(P['OUTPUT_ROOT'])
TMP = Path(P['TMP_DIR'])
QC = Path(P['QC_DIR'])
BOB = Path(P['FAIR_TOP']) / 'BOB'
ANALYSIS_RUN = UTC.parent / 'analysis/notebook_workflow/45e939498e85'   # update after a new analysis run

STYLE = {'font.size': 8, 'font.family': 'sans-serif', 'axes.spines.top': False, 'axes.spines.right': False}
plt.rcParams.update(STYLE)
T0, T1 = pd.Timestamp('1991-04-20'), pd.Timestamp('1991-12-10')


def utc(s):
    return pd.to_datetime(s, utc=True, format='ISO8601').dt.tz_convert(None)


def save(fig, name, dpi=200):
    fig.savefig(FIG / f'{name}.pdf', metadata={'CreationDate': None})
    fig.savefig(FIG / f'{name}.png', dpi=dpi)
    plt.close(fig)
    print('wrote', name)


def copy(src, name):
    shutil.copy(src, FIG / name)
    print('copied', src.name, '->', name)


def tex(s):
    return (str(s).replace('\\', r'\textbackslash{}').replace('&', r'\&').replace('%', r'\%').replace('_', r'\_')
            .replace('#', r'\#').replace('$', r'\$').replace('~', r'\textasciitilde{}').replace('−', '$-$'))


def pipeline_definitions(notebook):
    from datetime import datetime, timezone
    ns = {'PIPELINE': PIPELINE, '__name__': 'supplement', 'np': np, 'pd': pd, 'plt': plt, 'json': json,
          'Path': Path, 'sys': sys, 'datetime': datetime, 'timezone': timezone}
    for c in json.loads((PIPELINE / notebook).read_text())['cells']:
        if 'pipeline_definitions' in c.get('metadata', {}).get('tags', []):
            exec(''.join(c['source']), ns)
    return ns

print(SUPP, UTC, ANALYSIS_RUN.name, sep='\n')

/Users/thompsong/Developer/Pinatubo1991SeismicData/papers/SRL/supplement
/Users/thompsong/work/Pinatubo1991Processing/utc
45e939498e85


## Figure S1: documented pre-eruption telemetry (step 70)

In [2]:
ns70 = pipeline_definitions('70_network_history_and_stationxml.ipynb')
with tempfile.TemporaryDirectory() as d:
    ns70['diagrams'](Path(d), notes=False)
    copy(Path(d) / 'telemetry.png', 'figS01_telemetry.png')
plt.rcParams.update(STYLE)

copied telemetry.png -> figS01_telemetry.png


## Table S1: radio-frequency maps (step 68)

Discriminator slot, channel, site, attenuation, VCO centre frequency and radio frequency or link, as written on the dated maps. `SEIS_MAP.CUR` repeats the 8 July map in `SEIS_MAP.OLD`; it is listed once.

In [3]:
r = pd.read_csv(BOB / 'radio/radio_records.csv', dtype=str)
r = r[r.source_file == 'SEIS_MAP.OLD'].copy()
r['date'] = pd.to_datetime(r.recorded_time).dt.strftime('%-d %b')
lines = []
for date, g in r.groupby('recorded_time', sort=True):
    lines.append(r'\multicolumn{7}{l}{\textbf{' + pd.Timestamp(date).strftime('%-d %B 1991') + r'}}\\')
    for x in g.itertuples():
        relay = x.relay_as_written if isinstance(x.relay_as_written, str) else ''
        lines.append(' & '.join(tex(v if isinstance(v, str) else '') for v in
                     [x.disc_slot, x.legacy_id, (string.capwords(x.site_as_written) if isinstance(x.site_as_written, str) else ''), x.attenuation_as_written, x.vco_as_written,
                      x.radio_as_written, relay]) + r'\\')
RADIO_HEAD = r'''{\footnotesize
\setlength{\tabcolsep}{3pt}
\begin{longtable}{lllllll}
\caption{Radio-frequency maps as written: discriminator slot, channel, site, attenuation (dB), VCO centre frequency (Hz), radio frequency (MHz) or link, and relay. A ditto mark (") repeats the line above. The \file{SEIS\_MAP.CUR} copy of the 8 July map is identical and is not repeated.}\label{tab:radio}\\
\toprule
Slot & Channel & Site & Atten. & VCO & Radio & Relay\\
\midrule\endfirsthead
\toprule
Slot & Channel & Site & Atten. & VCO & Radio & Relay\\
\midrule\endhead
\bottomrule\endfoot
'''
(TAB / 'radio_maps.tex').write_text(RADIO_HEAD + '\n'.join(lines) + '\n\\end{longtable}\n}\n')
print(len(r), 'rows;', r.recorded_time.unique().tolist())

28 rows; ['1991-05-14', '1991-06-27', '1991-07-08']


## Table S2: HYPO71 velocity model, and the control file

The layered model is read from the recovered June 1991 control cards; the whole file is reproduced verbatim in the supplement, because the station-card fields after the coordinates are not documented here.

In [4]:
txt = (PIPELINE / 'hypo71_settings_199106.txt').read_text().splitlines()
stations, model = [], []
for line in txt:
    m = re.match(r'^\s{2}([A-Z0-9]{4})(\d{2})(\d{2}\.\d{2})\s+(\d{3})(\d{2}\.\d{2})(\d)?\s+(.*)$', line)
    if m:
        lat = int(m[2]) + float(m[3]) / 60
        lon = int(m[4]) + float(m[5]) / 60
        stations.append(dict(code=m[1], lat=lat, lon=lon, rest=m[7].split()))
        continue
    m = re.match(r'^\s*(\d+\.\d+)\s+(\d+\.\d+)\s*$', line)
    if m:
        model.append((float(m[1]), float(m[2])))
model_tex = [f'{v:.2f} & {z:.1f} & {(model[i + 1][1] if i + 1 < len(model) else float("nan")):.1f}'.replace('nan', r'$\infty$') + r'\\'
             for i, (v, z) in enumerate(model)]
(TAB / 'hypo71_model.tex').write_text('\\begin{tabular}{rrr}\n\\toprule\n$V_P$ (km/s) & Top (km) & Bottom (km)\\\\\n\\midrule\n'
                                     + '\n'.join(model_tex) + '\n\\bottomrule\n\\end{tabular}\n')
shutil.copy(PIPELINE / 'hypo71_settings_199106.txt', TAB / 'hypo71_settings_199106.txt')   # shown verbatim
print(model)
print(len(stations), 'station cards')

[(3.3, 0.0), (5.0, 1.0), (5.7, 4.0), (6.7, 15.0), (8.0, 25.0)]
16 station cards


## Table S3: clock rules (`clock_regimes.csv`)

In [5]:
cr = pd.read_csv(PIPELINE / 'clock_regimes.csv', comment='#', dtype=str).fillna('')
rows = []
for x in cr.itertuples():
    rule = [tex(x.stream.replace(';', '; ')), tex(x.regime).replace(r'\_', r'\_\allowbreak{}'), tex(x.clock), tex(x.start.replace('T', ' ')), tex(x.end.replace('T', ' ')),
            tex(' to '.join(v for v in [x.seq_min, x.seq_max] if v) or ''), tex(x.marker), tex(x.utc_offset_h), tex(x.uncertainty_s),
            tex(x.confidence), tex(x.evidence)]
    rows.append(' & '.join(rule) + r'\\')
CLOCK_HEAD = r'''\begin{landscape}
{\scriptsize
\setlength{\tabcolsep}{3pt}
\begin{longtable}{>{\raggedright}p{2.3cm}>{\raggedright}p{2.6cm}c>{\raggedright}p{1.6cm}>{\raggedright}p{1.6cm}>{\raggedright}p{2.7cm}rrrl>{\raggedright\arraybackslash}p{6.6cm}}
\caption{Clock rules (\file{10\_conversion/clock\_regimes.csv}). Start and end are on the stream's own raw clock, half-open; offsets in hours (UTC = raw + offset), uncertainties in seconds. Sequence and marker apply only around the 26 May reset.}\label{tab:clockrules}\\
\toprule
Stream & Rule & Clock & Start & End & Sequence & Marker & Offset & Unc. & Conf. & Evidence\\
\midrule\endfirsthead
\toprule
Stream & Rule & Clock & Start & End & Sequence & Marker & Offset & Unc. & Conf. & Evidence\\
\midrule\endhead
\bottomrule\endfoot
'''
(TAB / 'clock_rules.tex').write_text(CLOCK_HEAD + '\n'.join(rows) + '\n\\end{longtable}\n}\n\\end{landscape}\n')
print(len(rows), 'rules')

27 rules


## Figure S2: EVNT trigger counts against waveform triggers, by week

For each week, hourly EVNT counts (summed over channels) are cross-correlated with hourly counts of triggered waveform records at lags of ±12 h, using square-root counts with a 24-h running mean removed so that slow changes in activity do not dominate. On the raw clocks the ridge of high correlation sits at −7 h before the 26 May reset (both PCs on Philippine time, but H = wall clock + 7 h), at +1 h while the RSAM PC was on Philippine time (UTC+8) and the DMX header clock H on UTC+7, at −7 h once the RSAM PC moved to GMT on 13 September, and at −8 h after daylight time ended on 27 October. After the clock rules it is at 0 h throughout.

In [6]:
ev = pd.read_csv(BOB / 'rsam_events/rsam_events_records.csv',
                 usecols=['recorded_time', 'raw_recorded_time', 'value_01', 'qc_use_for_coverage'], low_memory=False)
ev = ev[ev.qc_use_for_coverage & ev.value_01.notna()]
wf = pd.read_csv(TMP / '10_waveform_index.csv', usecols=['starttime', 'raw_starttime'], low_memory=False)
HOURS = pd.date_range('1991-04-28', '1991-12-01', freq='h')
LAGS = np.arange(-12, 13)

def lagogram(ev_t, wf_t):
    e = np.sqrt(pd.Series(ev.value_01.to_numpy(), index=ev_t.dt.floor('h')).groupby(level=0).sum().reindex(HOURS, fill_value=0))
    w = np.sqrt(wf_t.dt.floor('h').value_counts().reindex(HOURS, fill_value=0))
    # Remove day-scale changes in activity so the correlation measures hour-scale alignment.
    e, w = e - e.rolling(25, center=True, min_periods=1).mean(), w - w.rolling(25, center=True, min_periods=1).mean()
    weeks, rows = [], []
    for wk, g in pd.Series(HOURS, index=HOURS).groupby(pd.Grouper(freq='7D')):
        sl = slice(g.index[0], g.index[-1])
        with np.errstate(all='ignore'):
            rows.append([e.shift(-int(l))[sl].corr(w[sl]) for l in LAGS])
        weeks.append(wk)
    return pd.DatetimeIndex(weeks), np.array(rows, float)

fig, ax = plt.subplots(2, 1, figsize=(7.2, 4.0), sharex=True)
for a, (lab, et, wt) in zip(ax, [('Raw clocks', pd.to_datetime(ev.raw_recorded_time), utc(wf.raw_starttime)),
                                  ('UTC (clock rules applied)', pd.to_datetime(ev.recorded_time), utc(wf.starttime))]):
    weeks, r = lagogram(et, wt)
    x = mdates.date2num(weeks)
    im = a.pcolormesh(np.r_[x, x[-1] + 7], np.r_[LAGS - 0.5, LAGS[-1] + 0.5], r.T, cmap='magma', vmin=0, vmax=0.5, shading='flat')
    a.set_ylabel('Lag (h)'); a.set_yticks([-12, -8, -4, 0, 4, 8, 12]); a.set_title(lab, loc='left', fontsize=8)
    for dte in ['1991-05-26', '1991-09-13', '1991-10-27']:
        a.axvline(pd.Timestamp(dte), color='w', ls=':', lw=0.8)
    best = LAGS[np.nanargmax(np.nan_to_num(r, nan=-1), axis=1)]
    print(lab, dict(zip(weeks.strftime('%d %b'), best)))
fig.colorbar(im, ax=ax, label='Correlation')
ax[-1].xaxis.set_major_formatter(mdates.DateFormatter('%d %b')); ax[-1].set_xlabel('Week, 1991')
save(fig, 'figS02_evnt_trigger_lag')

/Users/thompsong/miniforge3/envs/flovopy_asl/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3015: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/Users/thompsong/miniforge3/envs/flovopy_asl/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3015: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)


Raw clocks {'28 Apr': np.int64(-11), '05 May': np.int64(0), '12 May': np.int64(-7), '19 May': np.int64(-7), '26 May': np.int64(1), '02 Jun': np.int64(1), '09 Jun': np.int64(1), '16 Jun': np.int64(1), '23 Jun': np.int64(1), '30 Jun': np.int64(1), '07 Jul': np.int64(-1), '14 Jul': np.int64(-3), '21 Jul': np.int64(1), '28 Jul': np.int64(1), '04 Aug': np.int64(2), '11 Aug': np.int64(1), '18 Aug': np.int64(-10), '25 Aug': np.int64(10), '01 Sep': np.int64(1), '08 Sep': np.int64(-9), '15 Sep': np.int64(-7), '22 Sep': np.int64(-7), '29 Sep': np.int64(1), '06 Oct': np.int64(4), '13 Oct': np.int64(9), '20 Oct': np.int64(4), '27 Oct': np.int64(-8), '03 Nov': np.int64(-12), '10 Nov': np.int64(-5), '17 Nov': np.int64(1), '24 Nov': np.int64(-12), '01 Dec': np.int64(-12)}
UTC (clock rules applied) {'28 Apr': np.int64(-11), '05 May': np.int64(0), '12 May': np.int64(0), '19 May': np.int64(0), '26 May': np.int64(0), '02 Jun': np.int64(0), '09 Jun': np.int64(0), '16 Jun': np.int64(0), '23 Jun': np.int64(

wrote figS02_evnt_trigger_lag


## Figures S3 and S4: all six clock-anchor records (step 72)

In [7]:
ns72 = pipeline_definitions('72_external_clock_anchors.ipynb')
m = pd.read_csv(UTC / 'clock_anchor_check/waveform_matches.csv')
m['origin'] = pd.to_datetime(m.origin, utc=True); m['p_arrival'] = pd.to_datetime(m.p_arrival, utc=True)
idx = pd.read_csv(TMP / '10_waveform_index.csv', usecols=['event_id', 'wav_file'], low_memory=False)
for name, recs in [('figS03_anchors_may', ['9105220C', '9105300K', '9105301M']),
                   ('figS04_anchors_aug_sep', ['91080826', '91081528', '9109055G'])]:
    rows = m[m.record.isin(recs)].drop_duplicates('record').set_index('record').loc[recs].reset_index()
    with plt.rc_context({'font.size': 7}):
        fig = ns72['plot_records'](rows, idx, (0.5, 3.0), (15.13, 120.35))
        fig.set_size_inches(7.2, 7.0); fig.tight_layout()
        save(fig, name, dpi=170)
    display(rows[['record', 'origin', 'mag', 'dist_km', 'depth_km', 'predicted_p_in_record_s']])

wrote figS03_anchors_may


,record,origin,mag,dist_km,depth_km,predicted_p_in_record_s
0,9105220C,1991-05-21 11:00:22.790000+00:00,6.60,2602.0,32.7378,49.0
1,9105300K,1991-05-30 06:14:07.230000+00:00,4.55,591.0,100.0000,1.5
2,9105301M,1991-05-30 13:17:43.700000+00:00,7.00,7863.0,33.5378,-1.0


wrote figS04_anchors_aug_sep


,record,origin,mag,dist_km,depth_km,predicted_p_in_record_s
0,91080826,1991-08-08 13:22:31.170000+00:00,5.0,351.0,11.7483,33.0
1,91081528,1991-08-15 13:36:01.920000+00:00,6.3,6272.0,186.8928,-32.9
2,9109055G,1991-09-05 20:30:44.170000+00:00,4.9,420.0,36.0000,4.2


## Figure S5: waveform availability by channel (UTC days with at least one trace)

In [8]:
tr = pd.read_csv(ANALYSIS_RUN / 'trace_inventory.csv', usecols=['station', 'channel', 'recorded_starttime', 'is_waveform'], low_memory=False)
tr = tr[tr.is_waveform]
tr['day'] = utc(tr.recorded_starttime).dt.floor('D')
tr['id'] = tr.station + '.' + tr.channel
order = ['UBO', 'PIE', 'PPO', 'BUR', 'GRN', 'BUG', 'CAB', 'CRW', 'DON', 'FNG', 'PI2', 'QAD']
ids = sorted(tr.id.unique(), key=lambda s: (order.index(s.split('.')[0]) if s.split('.')[0] in order else 99, s))
days = pd.date_range(tr.day.min(), tr.day.max(), freq='D')
mat = tr.groupby(['id', 'day']).size().unstack(fill_value=0).reindex(index=ids, columns=days, fill_value=0)
fig, ax = plt.subplots(figsize=(7.2, 3.4))
ax.imshow(np.log10(mat.values + 1), aspect='auto', cmap='Greys', interpolation='nearest',
          extent=[mdates.date2num(days[0]), mdates.date2num(days[-1] + pd.Timedelta(days=1)), len(ids) - 0.5, -0.5])
ax.set_yticks(range(len(ids)), ids, fontsize=6.5); ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b')); ax.set_xlabel('Date (UTC), 1991')
ax.axvline(mdates.date2num(pd.Timestamp('1991-06-15')), color='r', lw=0.7, ls='--')
fig.tight_layout()
save(fig, 'figS05_waveform_channels')
print(len(ids), 'channel identities;', len(tr), 'traces')

wrote figS05_waveform_channels
15 channel identities; 152933 traces


## Figures S6–S8: BOB products by series (UTC days with at least one usable value)

From each product's record table after conversion to UTC, using `qc_use_for_coverage`. COSPEC and the radio maps carry dates only; COSPEC dates are treated as Philippine days.

In [9]:
def availability(product, cols=('series', 'recorded_time', 'qc_use_for_coverage')):
    d = pd.read_csv(BOB / f'{product}/{product}_records.csv', usecols=list(cols), low_memory=False)
    d = d[d.qc_use_for_coverage.astype(bool)]
    d['series'] = d.series.replace({'.': product})   # rain is stored under series '.'
    d['day'] = pd.to_datetime(d.recorded_time, errors='coerce').dt.floor('D')
    d = d.dropna(subset=['day'])
    return d.groupby(['series', 'day']).size().unstack(fill_value=0)

def panel(ax, mat, title):
    days = pd.date_range(T0, T1, freq='D')
    mat = mat.reindex(columns=days, fill_value=0).sort_index()
    ax.imshow(mat.values > 0, aspect='auto', cmap='Greys', interpolation='nearest', vmin=0, vmax=1,
              extent=[mdates.date2num(days[0]), mdates.date2num(days[-1] + pd.Timedelta(days=1)), len(mat) - 0.5, -0.5])
    ax.set_yticks(range(len(mat)), mat.index, fontsize=6)
    ax.set_title(title, loc='left', fontsize=8)
    ax.xaxis_date(); ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
    ax.axvline(mdates.date2num(pd.Timestamp('1991-06-15')), color='r', lw=0.6, ls='--')

groups = [('figS06_bob_rsam_evnt', [('rsam', 'RSAM (10 min)'), ('rsam_events', 'EVNT counts (10 min)')]),
          ('figS07_bob_ssam', [('ssam', 'SSAM (1 min)'), ('ssam_data', 'SSAM DATA (1 min)')]),
          ('figS08_bob_other', [('tilt', 'Tilt (10 min)'), ('rain', 'Rain'), ('mudflow', 'Mudflow (AFM)'),
                                ('cospec', 'COSPEC (daily)'), ('counts', 'Hand-entered counts (hourly CAB, daily PIE)')])]
for name, prods in groups:
    mats = [(availability(p), t) for p, t in prods]
    heights = [max(len(m), 1) + 1.5 for m, _ in mats]
    fig, axs = plt.subplots(len(mats), 1, figsize=(7.2, 0.16 * sum(heights) + 0.8), sharex=True,
                            gridspec_kw=dict(height_ratios=heights))
    for ax, (m, t) in zip(np.atleast_1d(axs), mats):
        panel(ax, m, t)
    np.atleast_1d(axs)[-1].set_xlabel('Date (UTC), 1991')
    fig.tight_layout()
    save(fig, name)

wrote figS06_bob_rsam_evnt


wrote figS07_bob_ssam


wrote figS08_bob_other


## Figure S9: the two HYPO71 summaries for the same events

For groups with an origin from both `Pinatubo_all.sum` and `PINAALL.DAT`: differences in origin time, epicenter, depth and magnitude (PINAALL minus Pinatubo_all.sum).

In [10]:
o = pd.read_csv(TMP / '43_hypo71_origin_index.csv')
o['t'] = utc(o.origin_time)
src = sorted(o.source.unique()); print(src)
a = o[o.source == 'hypo40'].groupby('event_id').first()     # Pinatubo_all.sum
b = o[o.source == 'pinaall'].groupby('event_id').first()    # PINAALL.DAT
j = a.join(b, lsuffix='_a', rsuffix='_b', how='inner')
dt = (j.t_b - j.t_a).dt.total_seconds()
km = 111.2 * np.hypot(j.latitude_b - j.latitude_a, (j.longitude_b - j.longitude_a) * np.cos(np.radians(15.13)))
dz = j.depth_km_b - j.depth_km_a
dm = j.magnitude_b - j.magnitude_a
fig, ax = plt.subplots(1, 4, figsize=(7.2, 2.0))
for a_, v, lab, rng in [(ax[0], dt, 'Origin time (s)', (-3, 3)), (ax[1], km, 'Epicenter shift (km)', (0, 5)),
                        (ax[2], dz, 'Depth (km)', (-5, 5)), (ax[3], dm, 'Magnitude', (-0.5, 0.5))]:
    a_.hist(v.clip(*rng), bins=41, range=rng, color='0.4')
    a_.set_xlabel(lab); a_.set_yscale('log')
ax[0].set_ylabel('Groups')
fig.tight_layout()
save(fig, 'figS09_hypo_sources')
print(f'{len(j)} groups with both; identical (|dt|<0.01 s, <0.01 km, |dz|<0.01): '
      f'{int(((dt.abs() < 0.01) & (km < 0.01) & (dz.abs() < 0.01)).sum())}; median |dt| {dt.abs().median():.2f} s')

['hypo40', 'pinaall']


wrote figS09_hypo_sources
3500 groups with both; identical (|dt|<0.01 s, <0.01 km, |dz|<0.01): 757; median |dt| 0.40 s


## Figures S10–S11: pick and catalog quality control (copies of step 23 and 53 outputs)

In [11]:
copy(QC / 'step23_pick_event_diagnostics/23_box_ps_delay_by_station_merged.png', 'figS10_ps_by_station.png')
copy(QC / 'step53_seisan_rea_diagnostics/53_daily_counts.png', 'figS11_rea_daily_counts.png')

copied 23_box_ps_delay_by_station_merged.png -> figS10_ps_by_station.png
copied 53_daily_counts.png -> figS11_rea_daily_counts.png


## Figures S12–S13: analysis run (copies of analysis notebook figures)

In [12]:
A = ANALYSIS_RUN / 'figures'
copy(A / 'noise_windows_through_time.png', 'figS12_noise_windows.png')
copy(A / 'publication_7c8fc5c93414_00_spectrograms.png', 'figS13_example_spectrograms.png')
for f in ['noise_windows_through_time', 'publication_7c8fc5c93414_00_spectrograms']:
    print(f, '|', (A / f'{f}.caption.md').read_text().strip()[:300])

copied noise_windows_through_time.png -> figS12_noise_windows.png
copied publication_7c8fc5c93414_00_spectrograms.png -> figS13_example_spectrograms.png
noise_windows_through_time | Proposed initial-noise lengths, not measured acquisition pretrigger settings. Supported/shortened windows are unreviewed unless explicitly accepted. Grey minimum-only values are assumptions and are excluded from file medians. Components are collapsed to one median per station before file summaries; 
publication_7c8fc5c93414_00_spectrograms | DRAFT: fixed representative selection; author approval pending. XB.PIE..EHZ. Same PSD units and fixed colour limits (-10, 70 dB re 1 count²/Hz) across panels; values outside are colour-clipped only. Complete windows only; gaps remain missing. Window centres set times. Step is 0.16 s before sample ro


## Check

In [13]:
t = (SUPP / 'pinatubo_srl_supplement.tex').read_text()
used = set(re.findall(r'includegraphics(?:\[[^]]*\])?\{([^}]+)\}', t)) | set(re.findall(r'(?:input|verbatiminput)\{tables/([^}]+)\}', t))
present = {p.stem for p in FIG.iterdir()} | {p.stem for p in TAB.iterdir()}
used = {u.rsplit('.', 1)[0] if u.endswith(('.tex', '.txt')) else u for u in used}
print('missing:', sorted(used - present) or 'none')
print('made but unused:', sorted(present - used) or 'none')

missing: none
made but unused: none
